# SVM ABSA

#1. INSTALL AND IMPORT

In [ ]:
!pip -q install iterative-stratification

In [ ]:
from __future__ import annotations

import json
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import version

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score,
    make_scorer,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.svm import LinearSVC

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

SEED = 33
np.random.seed(SEED)

## 2. Configuration

There is a total of 12 combinations:

- weighting: TF or TF-IDF;
- lowercasing: off or on;
- n-gram range: (1,1), (1,2), (1,3).


In [ ]:
# ---- Data ----
DATA_PATH = "/content/annotations.json"

ASPECT_CATEGORIES = [
    "Baterija", "Kamera", "Ekran", "Memorija", "Zvučnici", "Izgled",
    "Hardver", "Softver", "Performanse", "Cena", "Opšta ocena"
]

# ---- Cross-validation ----
QUICK_MODE = False
OUTER_FOLDS = 3 if QUICK_MODE else 10
INNER_FOLDS = 2 if QUICK_MODE else 5
C_GRID = [0.1, 1.0] if QUICK_MODE else [0.01, 0.1, 1.0, 10.0]
N_JOBS = -1

# ---- Vectorization ----
MIN_DF = 1
MAX_DF = 1.0
MAX_FEATURES = None

# Primary metrics used to choose C inside nested CV.
ASPECT_TUNING_SCORER = "f1_macro"
POLARITY_TUNING_SCORER = "f1_macro"

OUTPUT_DIR = Path("/content/svm_absa_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load annotation data

In [ ]:
def _parse_aspect_list(value):
    """Normalize an aspect annotation value to a list of dictionaries."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        value = json.loads(value)
    return value


def load_annotation_data(path: str) -> pd.DataFrame:
    df = pd.read_json(path)

    if "phone" not in df.columns:
        df["phone"] = ""

    df = df.copy()
    df["comment"] = df["comment"].fillna("").astype(str).str.strip()
    df["phone"] = df["phone"].fillna("").astype(str).str.strip()
    df["aspect_categories"] = df["aspect_categories"].map(_parse_aspect_list)

    df = df[df["review_status"] != "NE"]

    df = df.reset_index(drop=True)
    df["review_id"] = np.arange(len(df), dtype=int)
    return df


df = load_annotation_data(DATA_PATH)
print("Reviews:", len(df))
df.head()

Reviews: 6454


,phone,comment,review_status,aspect_terms,aspect_categories,review_id
0,Huawei P50 Pro,"Najkonkretnije me zanima, da li na huawei tele...",DA,"[{'fr': 386, 'to': 394, 'trg': 'kvalitet', 'ca...","[{'category': 'Izgled', 'polarity': 'Pozitivan...",0
1,Huawei P50 Pro,Huawei je brend kvalitet i sve napravljeno da ...,DA,"[{'fr': 16, 'to': 24, 'trg': 'kvalitet', 'cate...","[{'category': 'Izgled', 'polarity': 'Pozitivan'}]",1
2,Huawei P50 Pro,"Pozdrav svima, da li je neko uspeo da resi pro...",DA,"[{'fr': 54, 'to': 68, 'trg': 'notifikacijama',...","[{'category': 'Softver', 'polarity': 'Negativa...",2
3,Huawei P50 Pro,"Nisam nigdje u Cg nasao zastitu za ekran ,inte...",DA,"[{'fr': 35, 'to': 40, 'trg': 'ekran', 'categor...","[{'category': 'Ekran', 'polarity': 'Negativan'...",3
4,Huawei P50 Pro,"Imam problem, ne mogu da kupim najobičniju apl...",DA,"[{'fr': 43, 'to': 53, 'trg': 'aplikaciju', 'ca...","[{'category': 'Softver', 'polarity': 'Negativa...",4


In [ ]:
def make_model_text(frame: pd.DataFrame) -> pd.Series:
    text = frame["comment"].fillna("").astype(str)
    return text


def clean_annotations_for_review(aspects, review_id=None):
    """Keep one polarity per category"""
    by_category = {}
    for item in aspects:
        if not isinstance(item, dict):
            continue
        category = item.get("category")
        polarity = item.get("polarity")
        if category not in ASPECT_CATEGORIES or polarity is None or str(polarity).strip() == "":
            continue
        polarity = str(polarity).strip()
        by_category[category] = polarity
    return by_category


def build_targets(frame: pd.DataFrame):
    aspect_sets = []
    polarity_rows = []
    model_texts = make_model_text(frame).reset_index(drop=True)

    for pos, (_, row) in enumerate(frame.iterrows()):
        by_category = clean_annotations_for_review(
            row["aspect_categories"], review_id=row["review_id"]
        )
        aspect_sets.append(list(by_category.keys()))
        model_text = model_texts.iloc[pos]
        for category, polarity in by_category.items():
            polarity_rows.append({
                "review_id": int(row["review_id"]),
                "text": model_text,
                "aspect": category,
                "polarity": polarity,
            })

    mlb = MultiLabelBinarizer(classes=ASPECT_CATEGORIES)
    y_aspect = mlb.fit_transform(aspect_sets)
    polarity_df = pd.DataFrame(polarity_rows)
    return y_aspect, polarity_df


X_text = make_model_text(df)
y_aspect, polarity_df = build_targets(df)

print("Aspect-polarity annotations:", len(polarity_df))

Aspect-polarity annotations: 14444


## 4. Dataset statistics


In [ ]:
aspect_counts = pd.Series(y_aspect.sum(axis=0), index=ASPECT_CATEGORIES, name="count").sort_values(ascending=False)
polarity_counts = polarity_df["polarity"].value_counts()

stats = pd.DataFrame({
    "reviews": [len(df)],
    "aspect_annotations": [len(polarity_df)],
})
display(stats)
display(aspect_counts.to_frame())
display(polarity_counts.to_frame("count"))

,reviews,aspect_annotations
0,6454,14444


,count
Opšta ocena,3099
Baterija,2199
Softver,1851
Kamera,1438
Hardver,1392
Performanse,1128
Ekran,968
Cena,858
Izgled,842
Zvučnici,505


,count
polarity,
Pozitivan,7743
Negativan,5742
Neutralan,483
Konflikt,476


## 5. Preprocess config and model pipeline creation


In [ ]:
@dataclass(frozen=True)
class PrepConfig:
    weighting: str       # "tf" or "tfidf"
    lowercase: bool
    max_ngram: int       # 1, 2, 3 means (1, max_ngram)

    @property
    def ngram_range(self):
        return (1, self.max_ngram)

    @property
    def name(self):
        lc = "lower-on" if self.lowercase else "lower-off"
        return f"{self.weighting}_{lc}_1-{self.max_ngram}gram"


def make_preprocessing_grid():
    return [
        #PrepConfig("tf", True, 3)
        PrepConfig(weighting=w, lowercase=lc, max_ngram=n)
        for w in ("tf", "tfidf")
        for lc in (False, True)
        #PrepConfig("tfidf", True, max_ngram=n)
        #for n in (1, 2, 3)
    ]


PREPROCESSING_CONFIGS = make_preprocessing_grid()
if QUICK_MODE:
    PREPROCESSING_CONFIGS = PREPROCESSING_CONFIGS[:2]

pd.DataFrame([asdict(c) | {"name": c.name} for c in PREPROCESSING_CONFIGS])

,weighting,lowercase,max_ngram,name
0,tfidf,True,1,tfidf_lower-on_1-1gram
1,tfidf,True,2,tfidf_lower-on_1-2gram
2,tfidf,True,3,tfidf_lower-on_1-3gram


In [ ]:
def make_vectorizer(config: PrepConfig) -> TfidfVectorizer:
    return TfidfVectorizer(
        lowercase=config.lowercase,
        ngram_range=config.ngram_range,
        use_idf=(config.weighting == "tfidf"),
        smooth_idf=True,
        sublinear_tf=False,
        norm="l2",
        min_df=MIN_DF,
        max_df=MAX_DF,
        max_features=MAX_FEATURES,
    )


def build_aspect_pipeline(config: PrepConfig, C: float = 1.0) -> Pipeline:
    return Pipeline([
        ("vectorizer", make_vectorizer(config)),
        ("clf", OneVsRestClassifier(
            LinearSVC(C=C, max_iter=20000)
        )),
    ])


def build_polarity_pipeline(config: PrepConfig, C: float = 1.0) -> Pipeline:
    features = ColumnTransformer([
        ("text", make_vectorizer(config), "text"),
        ("aspect", OneHotEncoder(handle_unknown="ignore"), ["aspect"]),
    ])
    return Pipeline([
        ("features", features),
        ("clf", LinearSVC(C=C, max_iter=20000)),
    ])

## 5. Evaluation metric calculation


In [ ]:
def multilabel_metrics(y_true, y_pred):
    return {
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }


def multiclass_metrics(y_true, y_pred):
    return {
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def summarize_cv(raw: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    rows = []
    keys = ["weighting", "lowercase", "max_ngram", "config_name"]
    for key_values, group in raw.groupby(keys, dropna=False):
        row = dict(zip(keys, key_values))
        for metric in metrics:
            row[f"{metric}_mean"] = group[metric].mean()
            row[f"{metric}_std"] = group[metric].std(ddof=1)
        modes = group["best_C"].mode()
        row["C_mode"] = modes.iloc[0] if len(modes) else np.nan
        row["mean_fit_seconds"] = group["fit_seconds"].mean()
        rows.append(row)
    return pd.DataFrame(rows).sort_values(f"{metrics[0]}_mean", ascending=False).reset_index(drop=True)


def safe_stratified_group_folds(y, groups, requested: int) -> int:
    tmp = pd.DataFrame({"y": np.asarray(y), "group": np.asarray(groups)})
    unique_groups_per_class = tmp.groupby("y")["group"].nunique()
    max_possible = int(unique_groups_per_class.min())
    n_splits = min(requested, max_possible)
    if n_splits < 2:
        raise ValueError(
            "Not enough distinct review groups in every polarity class for stratified grouped CV. "
            f"Minimum groups per class = {max_possible}."
        )
    if n_splits < requested:
        warnings.warn(
            f"Requested {requested} folds, but only {n_splits} are possible for this polarity split."
        )
    return n_splits

## 6. Cross validation — aspect category detection

Searching for the best preprocesor and hyperparameter values

In [ ]:
def evaluate_aspect_config_nested_cv(
    X: pd.Series,
    y: np.ndarray,
    config: PrepConfig,
    outer_folds: int = OUTER_FOLDS,
    inner_folds: int = INNER_FOLDS,
    c_grid: Iterable[float] = C_GRID,
):
    outer = MultilabelStratifiedKFold(
        n_splits=outer_folds, shuffle=True, random_state=SEED
    )
    rows = []

    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        inner = MultilabelStratifiedKFold(
            n_splits=inner_folds, shuffle=True, random_state=SEED + fold
        )

        grid = GridSearchCV(
            estimator=build_aspect_pipeline(config),
            param_grid={"clf__estimator__C": list(c_grid)},
            scoring=ASPECT_TUNING_SCORER,
            cv=inner,
            n_jobs=N_JOBS,
            refit=True,
        )

        t0 = time.perf_counter()
        grid.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - t0
        y_pred = grid.predict(X_test)

        row = {
            "fold": fold,
            "weighting": config.weighting,
            "lowercase": config.lowercase,
            "max_ngram": config.max_ngram,
            "config_name": config.name,
            "best_C": grid.best_params_["clf__estimator__C"],
            "fit_seconds": fit_seconds,
        }
        row.update(multilabel_metrics(y_test, y_pred))
        rows.append(row)

    return pd.DataFrame(rows)


def run_aspect_preprocessing_experiments(X, y, configs=PREPROCESSING_CONFIGS):
    all_rows = []
    for i, config in enumerate(configs, start=1):
        print(f"[{i}/{len(configs)}] Aspect experiment: {config.name}")
        fold_results = evaluate_aspect_config_nested_cv(X, y, config)
        all_rows.append(fold_results)
        print(f"  macro-F1 = {fold_results['macro_f1'].mean():.4f}")

    raw = pd.concat(all_rows, ignore_index=True)
    summary = summarize_cv(
        raw,
        metrics=["macro_f1", "micro_f1", "macro_precision", "micro_precision", "macro_recall", "macro_recall", "weighted_f1", "accuracy", "hamming_loss"],
    )
    return raw, summary

In [ ]:
# Full run: 12 preprocessing variants × nested CV.
aspect_cv_raw, aspect_cv_summary = run_aspect_preprocessing_experiments(X_text, y_aspect)

aspect_cv_raw.to_csv(OUTPUT_DIR / "aspect_cv_folds.csv", index=False)
aspect_cv_summary.to_csv(OUTPUT_DIR / "aspect_preprocessing_summary.csv", index=False)

display(aspect_cv_summary)


[1/12] Aspect experiment: tf_lower-off_1-1gram
  macro-F1 = 0.6775
[2/12] Aspect experiment: tf_lower-off_1-2gram
  macro-F1 = 0.6730
[3/12] Aspect experiment: tf_lower-off_1-3gram
  macro-F1 = 0.6786
[4/12] Aspect experiment: tf_lower-on_1-1gram
  macro-F1 = 0.6900
[5/12] Aspect experiment: tf_lower-on_1-2gram
  macro-F1 = 0.6909
[6/12] Aspect experiment: tf_lower-on_1-3gram
  macro-F1 = 0.6981
[7/12] Aspect experiment: tfidf_lower-off_1-1gram
  macro-F1 = 0.6816
[8/12] Aspect experiment: tfidf_lower-off_1-2gram
  macro-F1 = 0.6751
[9/12] Aspect experiment: tfidf_lower-off_1-3gram
  macro-F1 = 0.6714
[10/12] Aspect experiment: tfidf_lower-on_1-1gram
  macro-F1 = 0.6928
[11/12] Aspect experiment: tfidf_lower-on_1-2gram
  macro-F1 = 0.6930
[12/12] Aspect experiment: tfidf_lower-on_1-3gram
  macro-F1 = 0.6900


,weighting,lowercase,max_ngram,config_name,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,macro_precision_mean,macro_precision_std,...,macro_recall_mean,macro_recall_std,weighted_f1_mean,weighted_f1_std,accuracy_mean,accuracy_std,hamming_loss_mean,hamming_loss_std,C_mode,mean_fit_seconds
0,tf,True,3,tf_lower-on_1-3gram,0.698073,0.014071,0.761413,0.008665,0.789164,0.019370,...,0.652284,0.015432,0.754531,0.009477,0.378147,0.022911,0.093866,0.004480,10.0,85.385952
1,tfidf,True,2,tfidf_lower-on_1-2gram,0.693011,0.010531,0.757876,0.008358,0.806487,0.019706,...,0.633659,0.012200,0.749894,0.008967,0.381480,0.018180,0.093376,0.003822,10.0,45.755734
2,tfidf,True,1,tfidf_lower-on_1-1gram,0.692795,0.012833,0.741384,0.006890,0.787163,0.015703,...,0.635758,0.014173,0.736797,0.007137,0.348581,0.013641,0.099566,0.003324,10.0,15.663419
3,tf,True,2,tf_lower-on_1-2gram,0.690898,0.014465,0.752472,0.008159,0.803246,0.018200,...,0.628537,0.015971,0.745246,0.009030,0.374970,0.021433,0.094620,0.003876,10.0,45.518575
4,tfidf,True,3,tfidf_lower-on_1-3gram,0.690039,0.010833,0.757916,0.007158,0.774086,0.023063,...,0.657023,0.010895,0.749605,0.007798,0.373630,0.021019,0.097445,0.004111,10.0,86.159908
5,tf,True,1,tf_lower-on_1-1gram,0.689968,0.015473,0.741487,0.009607,0.778226,0.019449,...,0.633655,0.014119,0.737017,0.009968,0.353802,0.012443,0.099194,0.004372,10.0,17.252090
6,tfidf,False,1,tfidf_lower-off_1-1gram,0.681552,0.008207,0.735091,0.005989,0.787359,0.016276,...,0.620036,0.009367,0.729719,0.006279,0.347290,0.022658,0.101315,0.003672,10.0,16.423044
7,tf,False,3,tf_lower-off_1-3gram,0.678582,0.013530,0.749711,0.007822,0.782702,0.018313,...,0.631892,0.013118,0.741166,0.008604,0.363066,0.026066,0.098286,0.004546,10.0,86.440631
8,tf,False,1,tf_lower-off_1-1gram,0.677451,0.015459,0.733534,0.005802,0.782926,0.014123,...,0.616493,0.015040,0.728165,0.006867,0.345310,0.018476,0.101662,0.003199,10.0,20.587300
9,tfidf,False,2,tfidf_lower-off_1-2gram,0.675081,0.014227,0.748342,0.009822,0.805935,0.014774,...,0.612717,0.013417,0.738598,0.010810,0.374280,0.024463,0.096498,0.005150,10.0,49.007821


## 7. Cross validation — polarity classification

Searching for the best preprocesor and hyperparameter values

In [ ]:
def evaluate_polarity_config_nested_cv(
    polarity_data: pd.DataFrame,
    config: PrepConfig,
    outer_folds: int = OUTER_FOLDS,
    inner_folds: int = INNER_FOLDS,
    c_grid: Iterable[float] = C_GRID,
):
    X = polarity_data[["text", "aspect"]]
    y = polarity_data["polarity"].astype(str)
    groups = polarity_data["review_id"].to_numpy()

    actual_outer = safe_stratified_group_folds(y, groups, outer_folds)
    outer = StratifiedGroupKFold(
        n_splits=actual_outer, shuffle=True, random_state=SEED
    )

    rows = []
    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        g_train = groups[train_idx]

        actual_inner = safe_stratified_group_folds(y_train, g_train, inner_folds)
        inner = StratifiedGroupKFold(
            n_splits=actual_inner, shuffle=True, random_state=SEED + fold
        )

        grid = GridSearchCV(
            estimator=build_polarity_pipeline(config),
            param_grid={"clf__C": list(c_grid)},
            scoring=POLARITY_TUNING_SCORER,
            cv=inner,
            n_jobs=N_JOBS,
            refit=True,
        )

        t0 = time.perf_counter()
        grid.fit(X_train, y_train, groups=g_train)
        fit_seconds = time.perf_counter() - t0
        y_pred = grid.predict(X_test)

        row = {
            "fold": fold,
            "weighting": config.weighting,
            "lowercase": config.lowercase,
            "max_ngram": config.max_ngram,
            "config_name": config.name,
            "best_C": grid.best_params_["clf__C"],
            "fit_seconds": fit_seconds,
        }
        row.update(multiclass_metrics(y_test, y_pred))
        rows.append(row)

    return pd.DataFrame(rows)


def run_polarity_preprocessing_experiments(polarity_data, configs=PREPROCESSING_CONFIGS):
    all_rows = []
    for i, config in enumerate(configs, start=1):
        print(f"[{i}/{len(configs)}] Polarity experiment: {config.name}")
        fold_results = evaluate_polarity_config_nested_cv(polarity_data, config)
        all_rows.append(fold_results)
        print(f"  macro-F1 = {fold_results['macro_f1'].mean():.4f}")

    raw = pd.concat(all_rows, ignore_index=True)
    summary = summarize_cv(
        raw,
        metrics=["macro_f1", "micro_f1", "macro_precision", "micro_precision", "macro_recall", "micro_recall", "weighted_f1", "accuracy"],
    )
    return raw, summary

In [ ]:
polarity_cv_raw, polarity_cv_summary = run_polarity_preprocessing_experiments(polarity_df)

polarity_cv_raw.to_csv(OUTPUT_DIR / "polarity_cv_folds.csv", index=False)
polarity_cv_summary.to_csv(OUTPUT_DIR / "polarity_preprocessing_summary.csv", index=False)

display(polarity_cv_summary)

[1/3] Polarity experiment: tfidf_lower-on_1-1gram
  macro-F1 = 0.4049
[2/3] Polarity experiment: tfidf_lower-on_1-2gram
  macro-F1 = 0.4021
[3/3] Polarity experiment: tfidf_lower-on_1-3gram
  macro-F1 = 0.3986


,weighting,lowercase,max_ngram,config_name,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,macro_precision_mean,macro_precision_std,...,macro_recall_mean,macro_recall_std,micro_recall_mean,micro_recall_std,weighted_f1_mean,weighted_f1_std,accuracy_mean,accuracy_std,C_mode,mean_fit_seconds
0,tfidf,True,1,tfidf_lower-on_1-1gram,0.404870,0.007358,0.752659,0.016929,0.465070,0.066374,...,0.410291,0.005231,0.752659,0.016929,0.730384,0.017457,0.752659,0.016929,10.0,37.193520
1,tfidf,True,2,tfidf_lower-on_1-2gram,0.402101,0.007342,0.777005,0.014731,0.429864,0.090358,...,0.413408,0.006018,0.777005,0.014731,0.750023,0.016215,0.777005,0.014731,10.0,86.203854
2,tfidf,True,3,tfidf_lower-on_1-3gram,0.398590,0.006696,0.773176,0.014651,0.426007,0.083063,...,0.408997,0.005835,0.773176,0.014651,0.745280,0.016521,0.773176,0.014651,10.0,148.483372


## 8. Select the best preprocessing variants


In [ ]:
def config_from_summary_row(row) -> PrepConfig:
    return PrepConfig(
        weighting=str(row["weighting"]),
        lowercase=bool(row["lowercase"]),
        max_ngram=int(row["max_ngram"]),
    )

try:
    #BEST_ASPECT_CONFIG = config_from_summary_row(aspect_cv_summary.iloc[0])
    BEST_ASPECT_CONFIG = PrepConfig(weighting="tf", lowercase=True, max_ngram=3)
except NameError:
    BEST_ASPECT_CONFIG = PrepConfig(weighting="tf", lowercase=True, max_ngram=3)

try:
    #BEST_POLARITY_CONFIG = config_from_summary_row(polarity_cv_summary.iloc[0])
    BEST_POLARITY_CONFIG = PrepConfig(weighting="tf", lowercase=True, max_ngram=1)
except NameError:
    BEST_POLARITY_CONFIG = PrepConfig(weighting="tf", lowercase=True, max_ngram=1)


print("Best aspect preprocessing:", BEST_ASPECT_CONFIG)
print("Best polarity preprocessing:", BEST_POLARITY_CONFIG)

Best aspect preprocessing: PrepConfig(weighting='tf', lowercase=True, max_ngram=3)
Best polarity preprocessing: PrepConfig(weighting='tf', lowercase=True, max_ngram=1)


## 9. End-to-end Cross validation


In [ ]:
def _fit_best_aspect_on_train(X_train, y_train, config: PrepConfig, fold_seed: int):
    inner = MultilabelStratifiedKFold(
        n_splits=INNER_FOLDS, shuffle=True, random_state=fold_seed
    )
    grid = GridSearchCV(
        build_aspect_pipeline(config),
        {"clf__estimator__C": C_GRID},
        scoring=ASPECT_TUNING_SCORER,
        cv=inner,
        n_jobs=N_JOBS,
        refit=True,
    )
    grid.fit(X_train, y_train)
    return grid.best_estimator_, grid.best_params_["clf__estimator__C"]


def _fit_best_polarity_on_train(pol_train: pd.DataFrame, config: PrepConfig, fold_seed: int):
    X_train = pol_train[["text", "aspect"]]
    y_train = pol_train["polarity"].astype(str)
    groups = pol_train["review_id"].to_numpy()
    actual_inner = safe_stratified_group_folds(y_train, groups, INNER_FOLDS)
    inner = StratifiedGroupKFold(
        n_splits=actual_inner, shuffle=True, random_state=fold_seed
    )
    grid = GridSearchCV(
        build_polarity_pipeline(config),
        {"clf__C": C_GRID},
        scoring=POLARITY_TUNING_SCORER,
        cv=inner,
        n_jobs=N_JOBS,
        refit=True,
    )
    grid.fit(X_train, y_train, groups=groups)
    return grid.best_estimator_, grid.best_params_["clf__C"]

def count_svm_parameters(model):
    clf = model.named_steps["clf"]

    # ACD: OneVsRestClassifier
    if hasattr(clf, "estimators_"):
        return sum(
            estimator.coef_.size + estimator.intercept_.size
            for estimator in clf.estimators_
        )

    # ACSA: običan LinearSVC
    return clf.coef_.size + clf.intercept_.size

def truth_pair_set(aspects):
    by_category = clean_annotations_for_review(aspects)
    return {f"{category}|||{polarity}" for category, polarity in by_category.items()}


def predict_pair_sets_for_reviews(review_frame, aspect_model, polarity_model):
    review_text = make_model_text(review_frame)
    aspect_pred = aspect_model.predict(review_text)
    predicted_sets = [set() for _ in range(len(review_frame))]

    pair_rows = []
    pair_positions = []
    for pos, row_pred in enumerate(aspect_pred):
        for j, present in enumerate(row_pred):
            if present:
                pair_rows.append({"text": review_text.iloc[pos], "aspect": ASPECT_CATEGORIES[j]})
                pair_positions.append(pos)

    if pair_rows:
        pair_frame = pd.DataFrame(pair_rows)
        polarity_pred = polarity_model.predict(pair_frame[["text", "aspect"]])
        for pos, row, polarity in zip(pair_positions, pair_rows, polarity_pred):
            predicted_sets[pos].add(f"{row['aspect']}|||{polarity}")

    return aspect_pred, predicted_sets


def evaluate_end_to_end_nested_cv(
    frame: pd.DataFrame,
    y_aspects: np.ndarray,
    polarity_data: pd.DataFrame,
    aspect_config: PrepConfig,
    polarity_config: PrepConfig,
):
    X = make_model_text(frame)
    outer = MultilabelStratifiedKFold(
        n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED
    )

    pair_classes = sorted({
        f"{row.aspect}|||{row.polarity}" for row in polarity_data.itertuples()
    })
    pair_mlb = MultiLabelBinarizer(classes=pair_classes)
    pair_mlb.fit([[]])

    oof_aspect_pred = np.zeros_like(y_aspects)
    oof_pair_pred = [set() for _ in range(len(frame))]
    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y_aspects), start=1):
        print(f"End-to-end fold {fold}/{OUTER_FOLDS} : {datetime.now().strftime("%H:%M:%S")}")
        train_ids = set(frame.iloc[train_idx]["review_id"].astype(int))

        aspect_model, aspect_C = _fit_best_aspect_on_train(
            X.iloc[train_idx], y_aspects[train_idx], aspect_config, SEED + 100 + fold
        )

        pol_train = polarity_data[polarity_data["review_id"].isin(train_ids)].copy()
        polarity_model, polarity_C = _fit_best_polarity_on_train(
            pol_train, polarity_config, SEED + 200 + fold
        )

        test_frame = frame.iloc[test_idx].copy()
        aspect_pred, pair_pred_sets = predict_pair_sets_for_reviews(
            test_frame, aspect_model, polarity_model
        )
        oof_aspect_pred[test_idx] = aspect_pred
        for global_idx, pair_set in zip(test_idx, pair_pred_sets):
            oof_pair_pred[global_idx] = pair_set

        true_sets = [truth_pair_set(v) for v in test_frame["aspect_categories"]]
        y_pair_true = pair_mlb.transform(true_sets)
        y_pair_pred = pair_mlb.transform(pair_pred_sets)

        row = {
            "fold": fold,
            "aspect_C": aspect_C,
            "polarity_C": polarity_C,
        }
        for k, v in multilabel_metrics(y_aspects[test_idx], aspect_pred).items():
            row[f"aspect_{k}"] = v
        pair_metrics = multilabel_metrics(y_pair_true, y_pair_pred)
        for k, v in pair_metrics.items():
            row[f"pair_{k}"] = v
        fold_rows.append(row)

    print({datetime.now().strftime("%H:%M:%S")})

    fold_df = pd.DataFrame(fold_rows)
    true_pair_sets = [truth_pair_set(v) for v in frame["aspect_categories"]]
    y_pair_true_all = pair_mlb.transform(true_pair_sets)
    y_pair_pred_all = pair_mlb.transform(oof_pair_pred)

    return {
        "folds": fold_df,
        "oof_aspect_pred": oof_aspect_pred,
        "pair_classes": pair_classes,
        "oof_pair_pred_sets": oof_pair_pred,
        "y_pair_true": y_pair_true_all,
        "y_pair_pred": y_pair_pred_all,
    }

In [ ]:
end_to_end = evaluate_end_to_end_nested_cv(
    df,
    y_aspect,
    polarity_df,
    BEST_ASPECT_CONFIG,
    BEST_POLARITY_CONFIG,
)

end_to_end["folds"].to_csv(OUTPUT_DIR / "end_to_end_cv_folds.csv", index=False)
display(end_to_end["folds"])

print("\nMean end-to-end aspect-polarity pair metrics:")
pair_columns = [c for c in end_to_end["folds"].columns if c.startswith("pair_")]
display(end_to_end["folds"][pair_columns].agg(["mean", "std"]).T)

print("\nPer-aspect out-of-fold classification report:")
print(classification_report(
    y_aspect,
    end_to_end["oof_aspect_pred"],
    target_names=ASPECT_CATEGORIES,
    zero_division=0,
))

End-to-end fold 1/10 : 16:54:30
End-to-end fold 2/10 : 16:56:30
End-to-end fold 3/10 : 16:58:27
End-to-end fold 4/10 : 17:00:22
End-to-end fold 5/10 : 17:02:19
End-to-end fold 6/10 : 17:04:14
End-to-end fold 7/10 : 17:06:07
End-to-end fold 8/10 : 17:07:59
End-to-end fold 9/10 : 17:09:53
End-to-end fold 10/10 : 17:11:43
{'17:13:32'}


,fold,aspect_C,polarity_C,aspect_macro_precision,aspect_macro_recall,aspect_micro_precision,aspect_micro_recall,aspect_macro_f1,aspect_micro_f1,aspect_weighted_f1,...,aspect_hamming_loss,pair_macro_precision,pair_macro_recall,pair_micro_precision,pair_micro_recall,pair_macro_f1,pair_micro_f1,pair_weighted_f1,pair_accuracy,pair_hamming_loss
0,1,10.0,10.0,0.757801,0.646794,0.780741,0.729917,0.689246,0.754474,0.746722,...,0.093780,0.269543,0.239850,0.603704,0.564404,0.249300,0.583393,0.561574,0.282707,0.039781
1,2,10.0,10.0,0.783606,0.669080,0.784116,0.752078,0.708164,0.767762,0.763320,...,0.089815,0.322010,0.265862,0.602166,0.577562,0.275323,0.589608,0.563918,0.311278,0.039679
2,3,10.0,10.0,0.787976,0.631564,0.783296,0.721414,0.683796,0.751082,0.742609,...,0.098319,0.280890,0.235364,0.595937,0.548857,0.246233,0.571429,0.540434,0.266458,0.042320
3,4,10.0,10.0,0.782534,0.664030,0.807606,0.749481,0.709012,0.777459,0.771225,...,0.086981,0.277392,0.252708,0.621924,0.577163,0.259924,0.598708,0.570049,0.297840,0.039212
4,5,10.0,10.0,0.782708,0.670365,0.781069,0.748615,0.706065,0.764498,0.757286,...,0.094899,0.342110,0.263646,0.583815,0.559557,0.278771,0.571429,0.545217,0.280564,0.043175
5,6,10.0,10.0,0.813661,0.661727,0.784125,0.733010,0.709158,0.757706,0.752006,...,0.096779,0.328025,0.268424,0.601632,0.562413,0.282160,0.581362,0.555425,0.256693,0.041804
6,7,10.0,10.0,0.798433,0.635404,0.787425,0.728028,0.682500,0.756562,0.746976,...,0.094106,0.281997,0.238537,0.595060,0.550173,0.248793,0.571737,0.540458,0.272171,0.041389
7,8,10.0,10.0,0.815132,0.636847,0.814132,0.733564,0.694959,0.771751,0.764534,...,0.088924,0.324718,0.245185,0.619816,0.558478,0.263372,0.587550,0.559971,0.297972,0.040172
8,9,10.0,10.0,0.763937,0.640837,0.776799,0.731168,0.678440,0.753293,0.745696,...,0.101777,0.334118,0.263849,0.599119,0.563925,0.276736,0.580990,0.551373,0.268174,0.043215
9,10,10.0,10.0,0.805852,0.666191,0.791448,0.730104,0.719387,0.759539,0.754937,...,0.093283,0.350161,0.251256,0.580645,0.535640,0.272085,0.557235,0.537647,0.267281,0.042941



Mean end-to-end aspect-polarity pair metrics:


,mean,std
pair_macro_precision,0.311096,0.030242
pair_macro_recall,0.252468,0.012422
pair_micro_precision,0.600382,0.013213
pair_micro_recall,0.559817,0.012736
pair_macro_f1,0.265270,0.013612
pair_micro_f1,0.579344,0.011751
pair_weighted_f1,0.552607,0.011306
pair_accuracy,0.280114,0.017374
pair_hamming_loss,0.041369,0.001553



Per-aspect out-of-fold classification report:
              precision    recall  f1-score   support

    Baterija       0.90      0.93      0.92      2199
      Kamera       0.86      0.89      0.87      1438
       Ekran       0.79      0.73      0.76       968
    Memorija       0.76      0.23      0.35       164
    Zvučnici       0.90      0.59      0.72       505
      Izgled       0.76      0.51      0.61       842
     Hardver       0.75      0.59      0.66      1392
     Softver       0.72      0.67      0.70      1851
 Performanse       0.68      0.61      0.64      1128
        Cena       0.78      0.59      0.67       858
 Opšta ocena       0.76      0.83      0.79      3099

   micro avg       0.79      0.74      0.76     14444
   macro avg       0.79      0.65      0.70     14444
weighted avg       0.79      0.74      0.75     14444
 samples avg       0.73      0.73      0.70     14444



In [ ]:
end_to_end["folds"]

,fold,aspect_C,polarity_C,aspect_macro_precision,aspect_macro_recall,aspect_micro_precision,aspect_micro_recall,aspect_macro_f1,aspect_micro_f1,aspect_weighted_f1,...,aspect_hamming_loss,pair_macro_precision,pair_macro_recall,pair_micro_precision,pair_micro_recall,pair_macro_f1,pair_micro_f1,pair_weighted_f1,pair_accuracy,pair_hamming_loss
0,1,10.0,10.0,0.808074,0.654110,0.806402,0.721200,0.714146,0.761425,0.756673,...,0.089692,0.335617,0.264992,0.644817,0.576687,0.284028,0.608852,0.580049,0.302083,0.036763
1,2,10.0,10.0,0.847106,0.672971,0.822956,0.743729,0.732014,0.781339,0.775880,...,0.088320,0.320050,0.259046,0.640660,0.578983,0.272757,0.608262,0.575375,0.299051,0.039557
2,3,10.0,10.0,0.785899,0.635876,0.803125,0.698370,0.696931,0.747093,0.741272,...,0.100274,0.338910,0.247853,0.631250,0.548913,0.269413,0.587209,0.558134,0.270998,0.040916
3,4,10.0,10.0,0.813853,0.647497,0.830495,0.728940,0.708374,0.776411,0.768524,...,0.086969,0.372139,0.270875,0.655573,0.575408,0.289603,0.612880,0.577478,0.315789,0.037644
4,5,10.0,10.0,0.816852,0.647115,0.794910,0.722449,0.705541,0.756949,0.750828,...,0.094082,0.320841,0.253643,0.628743,0.571429,0.270674,0.598717,0.565029,0.297420,0.038833
5,6,10.0,10.0,0.827311,0.618760,0.816901,0.710204,0.679024,0.759825,0.751576,...,0.093023,0.318683,0.237501,0.615023,0.534694,0.254699,0.572052,0.540731,0.272868,0.041438
6,7,10.0,10.0,0.810733,0.648415,0.820849,0.720678,0.710258,0.767509,0.761413,...,0.092635,0.326208,0.255147,0.640154,0.562034,0.272746,0.598556,0.563260,0.280063,0.039988
7,8,10.0,10.0,0.817695,0.635428,0.817832,0.723317,0.697659,0.767677,0.760300,...,0.091620,0.314791,0.247054,0.623367,0.551326,0.265032,0.585137,0.553948,0.289515,0.040902
8,9,10.0,10.0,0.793549,0.645041,0.808656,0.723014,0.703260,0.763441,0.758162,...,0.090090,0.298047,0.253321,0.656036,0.586558,0.268584,0.619355,0.589608,0.300300,0.036241
9,10,10.0,10.0,0.785826,0.637348,0.791101,0.715065,0.690387,0.751164,0.743869,...,0.094442,0.275822,0.238868,0.622926,0.563054,0.250486,0.591479,0.561734,0.288490,0.038762


## 10. Train final models on all available data

Nested CV above is used for unbiased evaluation. For deployment, the selected preprocessing configurations are retained and `C` is tuned once more on the complete annotated dataset before fitting the final estimators.

In [ ]:
def fit_final_models(frame, y_aspects, polarity_data, aspect_config, polarity_config):
    X = make_model_text(frame)

    aspect_cv = MultilabelStratifiedKFold(
        n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED + 500
    )
    aspect_grid = GridSearchCV(
        build_aspect_pipeline(aspect_config),
        {"clf__estimator__C": C_GRID},
        scoring=ASPECT_TUNING_SCORER,
        cv=aspect_cv,
        n_jobs=N_JOBS,
        refit=True,
    )
    aspect_grid.fit(X, y_aspects)

    pol_X = polarity_data[["text", "aspect"]]
    pol_y = polarity_data["polarity"].astype(str)
    pol_groups = polarity_data["review_id"].to_numpy()
    pol_folds = safe_stratified_group_folds(pol_y, pol_groups, OUTER_FOLDS)
    polarity_cv = StratifiedGroupKFold(
        n_splits=pol_folds, shuffle=True, random_state=SEED + 600
    )
    polarity_grid = GridSearchCV(
        build_polarity_pipeline(polarity_config),
        {"clf__C": C_GRID},
        scoring=POLARITY_TUNING_SCORER,
        cv=polarity_cv,
        n_jobs=N_JOBS,
        refit=True,
    )
    polarity_grid.fit(pol_X, pol_y, groups=pol_groups)

    return {
        "aspect_model": aspect_grid.best_estimator_,
        "polarity_model": polarity_grid.best_estimator_,
        "aspect_C": aspect_grid.best_params_["clf__estimator__C"],
        "polarity_C": polarity_grid.best_params_["clf__C"],
    }


final_models = fit_final_models(
    df,
    y_aspect,
    polarity_df,
    BEST_ASPECT_CONFIG,
    BEST_POLARITY_CONFIG,
)
print("Final aspect C:", final_models["aspect_C"])
print("Final polarity C:", final_models["polarity_C"])
print("Final aspect model param count:" ,count_svm_parameters(final_models["aspect_model"]))
print("Final polarity model param count:" ,count_svm_parameters(final_models["polarity_model"]))

Final aspect C: 10.0
Final polarity C: 10.0
Final aspect model param count: 1800216
Final polarity model param count: 654184


## 11. Prediction Function

In [ ]:
def predict_absa(
    comments: str | list[str],
    phones: str | list[str] | None = None,
    models=final_models,
) -> pd.DataFrame:
    if isinstance(comments, str):
        comments = [comments]
    comments = list(comments)

    if phones is None:
        phones = [""] * len(comments)
    elif isinstance(phones, str):
        phones = [phones] * len(comments)
    else:
        phones = list(phones)

    if len(phones) != len(comments):
        raise ValueError("phones and comments must have the same length")

    input_df = pd.DataFrame({"phone": phones, "comment": comments})
    input_df["review_id"] = np.arange(len(input_df))

    aspect_pred, pair_sets = predict_pair_sets_for_reviews(
        input_df,
        models["aspect_model"],
        models["polarity_model"],
    )

    rows = []
    for i, pair_set in enumerate(pair_sets):
        for pair in sorted(pair_set):
            aspect, polarity = pair.split("|||", 1)
            rows.append({
                "input_index": i,
                "aspect": aspect,
                "polarity": polarity,
            })
    return pd.DataFrame(rows, columns=["input_index", "aspect", "polarity"])


# Example:
predict_absa("Baterija traje odlično, ali je kamera veoma loša.")

,input_index,aspect,polarity
0,0,Baterija,Pozitivan
1,0,Kamera,Pozitivan


## 12. Save models and all experiment tables

The saved object contains both trained models, the selected preprocessing settings, the aspect list, and the chosen `C` values.

In [ ]:
artifact = {
    "aspect_model": final_models["aspect_model"],
    "polarity_model": final_models["polarity_model"],
    "aspect_categories": ASPECT_CATEGORIES,
    "best_aspect_config": asdict(BEST_ASPECT_CONFIG),
    "best_polarity_config": asdict(BEST_POLARITY_CONFIG),
    "aspect_C": final_models["aspect_C"],
    "polarity_C": final_models["polarity_C"]
}

model_path = OUTPUT_DIR / "svm_absa_models.joblib"
joblib.dump(artifact, model_path)
print("Saved:", model_path)
print("Results directory:", OUTPUT_DIR)

Saved: /content/svm_absa_results/svm_absa_models.joblib
Results directory: /content/svm_absa_results


In [ ]:
import psutil

# Get virtual memory details
input_memory = psutil.virtual_memory()

# Convert bytes to Gigabytes (GB) for readability
total_gb = input_memory.total / (1024 ** 3)
available_gb = input_memory.available / (1024 ** 3)
used_gb = input_memory.used / (1024 ** 3)

print(f"Total RAM: {total_gb:.2f} GB")
print(f"Available RAM: {available_gb:.2f} GB")
print(f"Used RAM: {used_gb:.2f} GB")
print(f"RAM Usage: {input_memory.percent}%")

Total RAM: 12.67 GB
Available RAM: 10.66 GB
Used RAM: 1.73 GB
RAM Usage: 15.9%
